# El conjunto de datos googleplaystore.csv contiene información sobre aplicaciones de la Play Store. El objetivo es transformar estos datos "crudos" (que vienen con símbolos, nulos y textos) en un formato numérico y limpio que un modelo de Ciencia de Datos pueda procesar.
 Objetivos:
1.Limpiar inconsistencias (caracteres especiales en números).

2.Tratar la ausencia de datos (valores nulos).

3.Estandarizar magnitudes para comparaciones equitativas.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder #solo importar las cosas q vamos a ocupar
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

In [ ]:
import kagglehub
import os

# Descargar el dataset
path = kagglehub.dataset_download("lava18/google-play-store-apps")

# Cargar el dataset
df = pd.read_csv(os.path.join(path, "googleplaystore.csv"))


df.head()

In [ ]:
df.info()

In [ ]:
# Ver cuántos duplicados hay
print(f"Duplicados detectados: {df.duplicated().sum()}")

# Eliminamos los duplicados y nos quedamos con la primera aparición
df = df.drop_duplicates()

Eliminamos duplicados para no sesgar el análisis. Si una aplicación aparece 5 veces, le daría una importancia artificial a su categoría o calificación.

In [ ]:
print(df.isnull().sum())

Es una métrica crítica. En lugar de borrar las filas (perderíamos muchos datos), llenaremos los vacíos con la mediana , que es más robusta que el promedio ante valores extremos.

Tipo / Clasificación de contenido: Al ser muy pocos los nulos, podemos eliminarlos o llenarlos con la moda.

In [ ]:
# Llenamos Rating con la mediana
df['Rating'] = df['Rating'].fillna(df['Rating'].median())

# Eliminamos las filas que aún tengan nulos en otras columnas (son muy pocas)
df = df.dropna()

Para poder normalizar "Installs" y "Price", primero deben ser números reales, sin texto.

In [ ]:
# Limpiamos Installs: quitamos '+' y ',' y convertimos a entero
df['Installs'] = df['Installs'].str.replace('+', '').str.replace(',', '').astype(int)

# Limpiamos Price: quitamos '$' y convertimos a float
df['Price'] = df['Price'].str.replace('$', '').astype(float)

Las máquinas no entienden que "ART_AND_DESIGN" es una categoría. Usamos OneHotEncoder para convertir categorías en columnas de 0 y 1. Lo aplicaremos a Category Type.

Las instalaciones pueden ser millones, mientras que el precio es pequeño (0 a 15 USD). La estandarización ajusta estos valores para que tengan media 0 y desviación estándar 1 , evitando que una variable "domine" a la otra por su tamaño.

In [ ]:
# Definimos qué columnas transformar
cols_categoricas = ['Category', 'Type']
cols_numericas = ['Installs', 'Price', 'Rating']

# Creamos el transformador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), cols_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cols_categoricas)
    ])

# Aplicamos las transformaciones
df_procesado = preprocessor.fit_transform(df)

# Ver el resultado (se convierte en una matriz de números)
print("Forma del dataframe procesado:", df_procesado.shape)

Hasta el momento se hizo:

Limpieza: Se eliminaron duplicados y se imputaron nulos en Rating con la mediana para mantener la integridad estadística sin perder volumen de datos.

Transformación: Se utilizó StandardScaler porque las escalas de Installs y Price presentan escalas muy diferentes; sin esto, cualquier algoritmo pensaría que el precio no es importante frente a las millones de descargas.

Se utiliza OneHotEncoder para las categorías, ya que no tienen un orden lógico (no es que una categoría sea "mayor" que otra), permitiendo que el modelo trate cada una de forma independiente.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Hallazgo 1: Distribución de Ratings
plt.figure(figsize=(8,4))
sns.histplot(df['Rating'], bins=20, kde=True, color='skyblue')
plt.title('Distribución de Calificaciones de Apps')
plt.show()

La distribución de calificaciones se concentra entre 4.0 y 4.5, presentando un sesgo hacia valores altos, lo que indica que la mayoría de las aplicaciones tienen buenas evaluaciones.

In [ ]:
import matplotlib.pyplot as plt

# Contamos cuántas son gratis (precio 0) vs pagas
gratis = df[df['Price'] == 0].shape[0]
pagas = df[df['Price'] > 0].shape[0]

print(f"Apps Gratuitas: {gratis}")
print(f"Apps Pagas: {pagas}")

# Visualización rápida
plt.bar(['Gratis', 'Pagas'], [gratis, pagas], color=['green', 'red'])
plt.title('Comparación de Monetización')
plt.ylabel('Cantidad de Apps')
plt.show()

Monetización: Al limpiar Price, observamos que la gran mayoría de las aplicaciones son gratuitas ( Free), lo que justifica por qué la columna Type es tan importante

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Agrupamos por categoría y sumamos las instalaciones
top_categories = df.groupby('Category')['Installs'].sum().sort_values(ascending=False).head(10)

# Graficamos
plt.figure(figsize=(12, 6))
top_categories.plot(kind='bar', color='coral')

plt.title('Top 10 Categorías con Más Instalaciones')
plt.xlabel('Categoría')
plt.ylabel('Total de Instalaciones (en miles de millones)')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Las categorías GAME y COMMUNICATION concentran la mayor cantidad de instalaciones, lo que sugiere una alta demanda en estas áreas. Esto justifica el uso de técnicas como OneHotEncoding, ya que permite al modelo identificar correctamente la categoría de cada aplicación y capturar su impacto en el comportamiento de los usuarios.

In [ ]:
top_categorias_rating = df.groupby("Category")["Rating"].mean() \
                         .sort_values(ascending=False) \
                         .head(10)

top_categorias_rating

Observamos que algunas categorías con menor volumen presentan mayores ratings promedio, lo que podría indicar nichos con alta satisfacción de usuario

In [ ]:
# Validación final del dataset

print("Valores nulos por columna:")
print(df.isnull().sum())

print("\nCantidad de duplicados:", df.duplicated().sum())

print("\nTipos de datos:")
print(df.dtypes)

print("\nResumen estadístico:")
df.describe()

# Validación final de los datos:
Luego del proceso de limpieza y transformación, se verificó que el dataset no presenta valores nulos ni duplicados.

Además, los tipos de datos son consistentes y adecuados para el análisis, ya que las variables numéricas fueron correctamente convertidas y las variables categóricas transformadas según corresponde.

Por lo tanto, como equipo concluimos que el conjunto de datos se encuentra preparado para ser utilizado en etapas posteriores de análisis o modelamiento.